In [3]:
# load all useful libraries 

import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import CubicSpline
#from hyperparameters import * # you will need this for the hyperparameter scan if you are actually running the model with GPU 

import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
#import seaborn as sns

from sklearn.metrics import roc_curve, auc
from sklearn.model_selection import train_test_split

from pytorch_tabnet.tab_model import TabNetRegressor
from pytorch_tabnet.metrics import Metric

In [1]:
# make sure you have the files downloaded (from the readme link) and put it in the data directory 

sig_df = pd.read_csv('data/Jewel.csv')
bg_df = pd.read_csv('data/Pythia.csv')
herwig_df = pd.read_csv('data/Herwig.csv')
hydrocolbt_df = pd.read_csv('data/CoLBTHydro.csv')

# pandas dataframes are super useful!  

sig_X = sig_df.dropna().iloc[:,0:-1].values
sig_Y = sig_df.dropna().iloc[:,-1].values

bg_X = bg_df.dropna().iloc[:,0:-1].values
bg_Y = bg_df.dropna().iloc[:,-1].values

herwig_X = herwig_df.dropna().iloc[:,0:-1].values
herwig_Y = herwig_df.dropna().iloc[:,-1].values

hydrocolbt_X = hydrocolbt_df.dropna().iloc[:,0:-1].values
hydrocolbt_Y = hydrocolbt_df.dropna().iloc[:,-1].values

# the dropna is that sometimes there are zeros in the input files so we ignore those and take all rows, we define signal as jewel and pythia as bkg  

NameError: name 'pd' is not defined

In [2]:
# setup the test-train split 90-10 and also you have an option to shuffle the inputs 
sig_X_train, sig_X_test, sig_Y_train, sig_Y_test = train_test_split(sig_X, sig_Y, train_size=0.9, shuffle=False)
bg_X_train, bg_X_test, bg_Y_train, bg_Y_test = train_test_split(bg_X, bg_Y, train_size=0.9, shuffle=False)
herwig_X_train, herwig_X_test, herwig_Y_train, herwig_Y_test = train_test_split(herwig_X, herwig_Y, train_size=0.9, shuffle=False)
hydrocolbt_X_train, hydrocolbt_X_test, hydrocolbt_Y_train, hydrocolbt_Y_test = train_test_split(hydrocolbt_X, hydrocolbt_Y, train_size=0.9, shuffle=False)

NameError: name 'train_test_split' is not defined

In [6]:
# combine the iputs 
X_train = np.concatenate((sig_X_train,  bg_X_train, herwig_X_train, hydrocolbt_X_train), axis=0)
Y_train = np.concatenate((sig_Y_train,  bg_Y_train, herwig_Y_train, hydrocolbt_Y_train), axis=0)
X_test = np.concatenate((sig_X_test, bg_X_test, herwig_X_test, hydrocolbt_X_test), axis=0)
Y_test = np.concatenate((sig_Y_test, bg_Y_test, herwig_Y_test, hydrocolbt_Y_test), axis=0)

In [7]:
# Fix random seed to ensure reproducibility 
seed = 42

In [8]:
# only if you have a GPU on hand 
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [10]:
# these parameters are something that we got using the hyperparameter scans 
n_d = 8
n_a = 8
n_steps = 3
gamma = 1.3

In [11]:
# you can either load the model that we have in the published paper or you can train yourself! 
# take a look at the slides regarding what a tabnetregressor is 

model = TabNetRegressor(device_name=device, n_a=n_a, n_d=n_d, n_steps=n_steps, gamma=gamma, seed=seed)
# model.load_model('model/model.zip')

/opt/anaconda3/envs/work_py_ml/lib/python3.11/site-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


In [12]:
# train the model 
model.fit(X_train, Y_train.reshape(-1, 1), eval_set=[(X_test, Y_test.reshape(-1, 1))], batch_size=16384, max_epochs=10)

epoch 0  | loss: 1.03223 | val_0_mse: 0.25607 |  0:00:19s
epoch 1  | loss: 0.21961 | val_0_mse: 0.22856 |  0:00:39s
epoch 2  | loss: 0.19818 | val_0_mse: 0.22916 |  0:00:59s
epoch 3  | loss: 0.17716 | val_0_mse: 0.2308  |  0:01:20s
epoch 4  | loss: 0.15099 | val_0_mse: 0.2304  |  0:01:39s
epoch 5  | loss: 0.12718 | val_0_mse: 0.22319 |  0:01:58s
epoch 6  | loss: 0.11288 | val_0_mse: 0.23175 |  0:02:17s
epoch 7  | loss: 0.10567 | val_0_mse: 0.22199 |  0:02:38s
epoch 8  | loss: 0.10136 | val_0_mse: 0.22053 |  0:02:57s
epoch 9  | loss: 0.09862 | val_0_mse: 0.20482 |  0:03:17s
Stop training because you reached max_epochs = 10 with best_epoch = 9 and best_val_0_mse = 0.20482


/opt/anaconda3/envs/work_py_ml/lib/python3.11/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


In [13]:
model.save_model('model/test_model')

Successfully saved model at model/test_model.zip


'model/test_model.zip'